# DOCX Retrieval Chatbot (TF‑IDF)
This notebook builds a simple retrieval-based chatbot from a Word **.docx** document.

## What it does
1. Extracts text from a `.docx` file
2. Chunks it into passages
3. Builds a TF‑IDF index over chunks
4. Retrieves the most relevant passage(s) for each question

In [2]:
#install dependencies
!pip -q install python-docx scikit-learn

In [ ]:
import re
from dataclasses import dataclass
from typing import List

from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [1]:
def preprocess_text(text: str) -> str:
    """Normalize text for matching."""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text

def read_docx(path: str) -> List[str]:
    """Extract non-empty paragraph text from a .docx file"""
    doc = Document(path)
    paras = []
    for p in doc.paragraphs:
        t = (p.text or "").strip()
        if t:
            paras.append(t)
    return paras

def chunk_paragraphs(paragraphs: List[str], max_words: int = 140, overlap_words: int = 30) -> List[str]:
    """Create word-window chunks with overlap for retrieval."""
    words = []
    for p in paragraphs:
        words.extend(p.split())
        words.append("\n")

    flat_words = [w for w in words if w != "\n"]
    chunks = []
    start = 0
    n = len(flat_words)
    if n == 0:
        return chunks

    while start < n:
        end = min(n, start + max_words)
        chunk = " ".join(flat_words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap_words)

    return chunks

def truncate(text: str, max_chars: int = 200) -> str:
    t = re.sub(r"\s+", " ", text).strip()
    if len(t) <= max_chars:
        return t
    return t[: max_chars - 1].rstrip() + "…"


NameError: name 'List' is not defined

In [6]:
@dataclass
class RetrievalResult:
    idx: int
    score: float
    text: str

class DocxRetrievalChatbot:
    def __init__(self, chunks: List[str]):
        if not chunks:
            raise ValueError("No text chunks found. Is the .docx empty or image-only?")

        self.chunks_raw = chunks
        self.chunks_proc = [preprocess_text(c) for c in chunks]

        # TF-IDF over chunks (bigrams help phrased queries)
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self.matrix = self.vectorizer.fit_transform(self.chunks_proc)

    def search(self, query: str, top_k: int = 3) -> List[RetrievalResult]:
        q = preprocess_text(query)
        q_vec = self.vectorizer.transform([q])
        sims = cosine_similarity(q_vec, self.matrix).flatten()
        top_idx = sims.argsort()[-top_k:][::-1]
        return [RetrievalResult(int(i), float(sims[i]), self.chunks_raw[int(i)]) for i in top_idx]

    def reply(self, query: str, top_k: int = 3, min_conf: float = 0.30) -> str:
        hits = self.search(query, top_k=top_k)
        best = hits[0]

        if best.score < min_conf:
            return (
                "I couldn’t find a confident match in the document.\n"
                "Try rephrasing with keywords that appear in the doc (feature name, section title, error message, etc.)."
            )

        related = "\n".join([f"- (score={h.score:.2f}) {truncate(h.text, 140)}" for h in hits[1:]])
        out = f"{best.text}\n\nTop match score: {best.score:.2f}\n"
        if related:
            out += f"\nRelated passages:\n{related}"
        return out

def build_bot_from_docx(docx_path: str, max_words: int = 140, overlap_words: int = 30) -> DocxRetrievalChatbot:
    paragraphs = read_docx(docx_path)
    chunks = chunk_paragraphs(paragraphs, max_words=max_words, overlap_words=overlap_words)
    return DocxRetrievalChatbot(chunks)


## Build the chatbot from your Word document
Set `DOCX_PATH` to your `.docx` file name/path and run the cell.

In [7]:
DOCX_PATH = "../data/QA.docx"
bot = build_bot_from_docx(DOCX_PATH)
print(f"Loaded doc and built index over {len(bot.chunks_raw)} chunks.")

Loaded doc and built index over 6 chunks.


## Ask questions
Run the cell below to chat. Stop by interrupting the kernel (◼︎ / ⛔).

In [8]:
while True:
    q = input("> ").strip()
    if q.lower() in {"quit", "exit"}:
        print("Bye.")
        break
    print(bot.reply(q))
    print()

AI and Data Analytics: Questions and Answers 1. What is Artificial Intelligence (AI)? Artificial Intelligence refers to computer systems that can perform tasks that normally require human intelligence, such as learning, reasoning, and decision-making. 2. What is machine learning? Machine learning is a subset of AI that enables systems to learn patterns from data and improve performance without being explicitly programmed. 3. What is deep learning? Deep learning is a type of machine learning that uses multi-layered neural networks to analyze complex data patterns. 4. What is natural language processing (NLP)? Natural language processing is an AI field that allows computers to understand, interpret, and generate human language. 5. What is computer vision? Computer vision is an AI technology that enables machines to analyze and interpret images and videos. 6. What is data mining? Data mining is the process of

Top match score: 0.37

Related passages:
- (score=0.32) power to train and depl

KeyboardInterrupt: Interrupted by user

In [9]:
questions = [
    "What is AI?",
    "What is machine learning?",
    "What is generative AI?"
]

for q in questions:
    print("User:", q)
    print("Bot:", bot.reply(q))
    print()

User: What is AI?
Bot: AI and Data Analytics: Questions and Answers 1. What is Artificial Intelligence (AI)? Artificial Intelligence refers to computer systems that can perform tasks that normally require human intelligence, such as learning, reasoning, and decision-making. 2. What is machine learning? Machine learning is a subset of AI that enables systems to learn patterns from data and improve performance without being explicitly programmed. 3. What is deep learning? Deep learning is a type of machine learning that uses multi-layered neural networks to analyze complex data patterns. 4. What is natural language processing (NLP)? Natural language processing is an AI field that allows computers to understand, interpret, and generate human language. 5. What is computer vision? Computer vision is an AI technology that enables machines to analyze and interpret images and videos. 6. What is data mining? Data mining is the process of

Top match score: 0.37

Related passages:
- (score=0.32) 

## Chatbot Modification Report

### Objective
The goal of this task was to modify the existing document-retrieval chatbot so that it:
1. Uses a **custom Q&A dataset**
2. Displays **sample question–answer interactions**
3. Uses a **modified confidence threshold** when returning answers.

---

### Custom Q&A Dataset
A new Q&A dataset titled **"AI and Data Analytics: Questions and Answers"** was created.
The dataset contains **30 questions and answers related to Artificial Intelligence, machine learning, and data analytics**.

Examples of questions included in the dataset:

- What is Artificial Intelligence (AI)?
- What is machine learning?
- What is natural language processing?
- What is generative AI?

This dataset was stored in a `.docx` file and loaded into the chatbot using the document reader function:

```python
paragraphs = read_docx("AI_QA_Knowledge_Base.docx")
chunks = chunk_paragraphs(paragraphs)
```

The chatbot retrieves relevant passages from this document when responding to user queries.

---

### Sample Chatbot Interactions

Below are example interactions demonstrating how the chatbot retrieves answers from the document.

**Example 1**

User: What is AI?

Bot:
Artificial Intelligence refers to computer systems that can perform tasks that normally require human intelligence, such as learning, reasoning, and decision-making.

Top match score: **0.37**

---

**Example 2**

User: What is machine learning?

Bot:
Machine learning is a subset of AI that enables systems to learn patterns from data and improve performance without being explicitly programmed.

Top match score: **0.46**

---

**Example 3**

User: What is generative AI?

Bot:
Generative AI refers to models that can generate new content such as text, images, audio, or code.

Top match score: **0.30**

---

### Confidence Score Modification

The chatbot determines whether to return an answer based on a **confidence score (similarity score)** between the user query and document passages.

The original confidence threshold was:

```python
min_conf = 0.16
```

This value was increased to:

```python
min_conf = 0.30
```

Increasing the threshold ensures that the chatbot only returns responses when there is a **stronger match between the question and document content**, improving the relevance of retrieved answers.

---

### Conclusion

The chatbot was successfully modified to use a custom Q&A dataset, display example question–answer interactions, and apply a higher confidence threshold when retrieving answers.
These changes demonstrate how document-retrieval chatbots can be adapted to work with different knowledge bases while maintaining reliable response quality.